In [ ]:
# Cell 1
import sys
sys.path.append('..')

import joblib
import pandas as pd
from src.models.evaluate import evaluate_cv, plot_roc, plot_pr, classification_summary
from src.models.predict import load_model, predict
from src.utils import config

# Cell 2
bundle = load_model(config.BEST_MODEL_PATH)
train = pd.read_csv(config.FEATURES_TRAIN)
feats = bundle['features']
X, y = train[feats], train['TARGET']

oof, auc = evaluate_cv(bundle['model'], X, y)
print('Best model CV AUC:', round(auc, 5))

# Cell 3 — Curves
config.ensure_dirs()
plot_roc(y, oof, config.FIGURES_DIR / 'roc_curve.png')
plot_pr(y, oof, config.FIGURES_DIR / 'pr_curve.png')
print('Curves saved to', config.FIGURES_DIR)

# Cell 4 — Classification summary
summary = classification_summary(y, oof, threshold=0.5)
print('Confusion matrix:', summary['confusion_matrix'])
print(summary['report'])

# Cell 5 — Submission
test = pd.read_csv(config.FEATURES_TEST)
preds = predict(bundle, test)
sub = pd.DataFrame({'uid': test['uid'], 'TARGET': preds.values})
sub.to_csv(config.SUBMISSION_PATH, index=False)
print('saved ->', config.SUBMISSION_PATH)
sub.head()